In [2]:
import numpy as np
import pandas as pd

import plotly.express as px

In [3]:
data = np.load('../data/metr_la_new.npz', allow_pickle=True)

dataset = data['targets']
dataset_size = len(dataset)

for i in range(0, len(dataset)):
    for j in range(0, len(dataset[i])):
        if np.isnan(dataset[i][j]):
            dataset[i][j] = dataset[max(i - 1, 0)][j]

In [4]:
train_size = int(0.6 * dataset_size)
test_size =  int(0.2 * dataset_size)
vertice = 0
dataset_for_vertice = dataset[:, vertice]
coef = 6
pred_cnt = 12

In [5]:
import random
samples_train = [i for i in random.sample(range(coef, int(dataset_size * 0.7)), train_size)]

X_train = [dataset_for_vertice[i - coef: i] for i in samples_train]
X_train = np.array(X_train)

y_train = [dataset_for_vertice[i:i + pred_cnt] for i in samples_train]
y_train = np.array(y_train)

samples_test = [i for i in random.sample(range(int(dataset_size * 0.7), dataset_size - pred_cnt), test_size)]
X_test = [dataset_for_vertice[i - coef: i] for i in samples_test]
X_test = np.array(X_test)

y_test = [dataset_for_vertice[i:i + pred_cnt] for i in samples_test]
y_test = np.array(y_test)

In [6]:
print(X_train.shape, y_train.shape)

(20563, 6) (20563, 12)


In [7]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(loss_function='MultiRMSE', learning_rate=0.1, iterations=2000, depth = 4)
cat.fit(pd.DataFrame(X_train),pd.DataFrame(y_train))

<frozen importlib._bootstrap>:488: RuntimeWarning: numpy.ufunc size changed, may indicate binary incompatibility. Expected 216 from C header, got 232 from PyObject


0:	learn: 38.9308573	total: 62.5ms	remaining: 2m 4s
1:	learn: 37.0930346	total: 72.9ms	remaining: 1m 12s
2:	learn: 35.4976972	total: 83.1ms	remaining: 55.3s
3:	learn: 34.1462064	total: 92ms	remaining: 45.9s
4:	learn: 32.9781944	total: 103ms	remaining: 41.2s
5:	learn: 32.0164731	total: 115ms	remaining: 38.1s
6:	learn: 31.2019292	total: 128ms	remaining: 36.3s
7:	learn: 30.5311656	total: 140ms	remaining: 34.8s
8:	learn: 29.9394087	total: 151ms	remaining: 33.4s
9:	learn: 29.4641729	total: 161ms	remaining: 32.1s
10:	learn: 29.0646930	total: 173ms	remaining: 31.2s
11:	learn: 28.7008246	total: 185ms	remaining: 30.7s
12:	learn: 28.4196359	total: 196ms	remaining: 30s
13:	learn: 28.1683092	total: 208ms	remaining: 29.5s
14:	learn: 27.9635680	total: 219ms	remaining: 28.9s
15:	learn: 27.7703122	total: 230ms	remaining: 28.5s
16:	learn: 27.6120344	total: 241ms	remaining: 28.1s
17:	learn: 27.4879983	total: 253ms	remaining: 27.8s
18:	learn: 27.3831110	total: 267ms	remaining: 27.8s
19:	learn: 27.2853775

In [8]:
result = []
for x, y in zip(X_test, y_test):
    result.append(abs(np.array(cat.predict(x)) - y_test).mean())
result = np.array(result)
result.mean()

8.395087413728003

In [9]:
px.line(y = np.array(result))

In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn

device = "cpu" if torch.cuda.is_available() else "cpu"

X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(device)

X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(device)

tensor_data = TensorDataset(X_train, y_train)

dataloader_train = DataLoader(tensor_data, 
                              shuffle = True,
                              batch_size = 16)

/tmp/ipykernel_6062/3358257101.py:7: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/tmp/ipykernel_6062/3358257101.py:8: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/tmp/ipykernel_6062/3358257101.py:10: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/tmp/ipykernel_6062/3358257101.py:11: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

